# Governança de Dados no Databricks

## Visão Geral

Governança de Dados no Databricks refere-se ao controlo centralizado de acesso, auditoria e gestão de dados através do **Unity Catalog**. Permite definir quem pode aceder a quais dados e com que nível de permissão.

**Para certificação:** Unity Catalog é a solução de governança unificada — substitui o Hive Metastore legado.

---

## Hierarquia de Objetos

![Unity Catalog Object Model](https://learn.microsoft.com/en-us/azure/databricks/_static/images/unity-catalog/object-model.png)

### Níveis de Escopo

| Objeto | Escopo | Descrição |
|--------|--------|-----------|
| **METASTORE** | Mais alto | Container de todos os catálogos (1 por região) |
| **CATALOG** | Alto | Agrupa schemas, primeiro nível do namespace |
| **SCHEMA** | Médio | Agrupa tabelas, views, functions (= database) |
| **TABLE** | Baixo | Tabela managed ou external |
| **VIEW** | Baixo | View SQL sobre tabelas |
| **FUNCTION** | Baixo | UDF (User Defined Function) |
| **VOLUME** | Baixo | Storage para ficheiros não-tabulares |

### Namespace Completo

```sql
-- Formato: catalog.schema.object
SELECT * FROM sales_catalog.bronze.customers;

-- Equivalente com USE
USE CATALOG sales_catalog;
USE SCHEMA bronze;
SELECT * FROM customers;
```

---

## Sintaxe de Permissões

### GRANT — Conceder Permissões

```sql
GRANT privilege ON object_type object_name TO principal;
```

**Exemplos:**

```sql
-- Permissão SELECT numa tabela
GRANT SELECT ON TABLE sales.bronze.customers TO `user@company.com`;

-- Múltiplas permissões
GRANT SELECT, MODIFY ON TABLE sales.bronze.orders TO `data_team`;

-- Permissão em schema inteiro
GRANT USAGE ON SCHEMA sales.bronze TO `analysts`;

-- Todas as permissões
GRANT ALL PRIVILEGES ON TABLE sales.gold.report TO `admin_group`;
```

### REVOKE — Remover Permissões

```sql
REVOKE privilege ON object_type object_name FROM principal;
```

**Exemplos:**

```sql
-- Remover SELECT
REVOKE SELECT ON TABLE sales.bronze.customers FROM `user@company.com`;

-- Remover todas permissões
REVOKE ALL PRIVILEGES ON SCHEMA sales.bronze FROM `temp_user`;
```

### DENY — Negar Explicitamente

```sql
DENY privilege ON object_type object_name TO principal;
```

**Para certificação:** DENY tem precedência sobre GRANT — mesmo que o utilizador tenha permissão via grupo, DENY bloqueia.

### SHOW GRANTS — Visualizar Permissões

```sql
-- Ver permissões num objecto
SHOW GRANTS ON TABLE sales.bronze.customers;

-- Ver permissões de um principal
SHOW GRANTS TO `user@company.com`;

-- Ver próprias permissões
SHOW GRANTS ON SCHEMA sales.bronze;
```

---

### Descrição de Cada Privilege

| Privilege | Descrição |
|-----------|-----------|
| **SELECT** | Ler dados de tabela ou view |
| **MODIFY** | INSERT, UPDATE, DELETE, MERGE em tabela |
| **CREATE** | Criar objectos dentro do schema/catalog |
| **USAGE** | Acesso ao container (obrigatório para navegar) |
| **READ_METADATA** | Ver metadados (DESCRIBE, SHOW) |
| **EXECUTE** | Executar função (UDF) |
| **READ FILES** | Ler ficheiros de um volume |
| **WRITE FILES** | Escrever ficheiros num volume |
| **ALL PRIVILEGES** | Todas as permissões aplicáveis ao objecto |

---

## USAGE — Conceito Crítico

### O que é USAGE?

USAGE é a permissão para **aceder a um container** (catalog ou schema). Sem USAGE, não é possível ver ou usar objectos dentro do container, mesmo tendo outras permissões.

### Cadeia de USAGE

Para aceder a uma tabela, o utilizador precisa:

```
USAGE no Catalog
    +
USAGE no Schema
    +
SELECT na Table
```

**Exemplo completo:**

```sql
-- Utilizador precisa das 3 permissões para fazer SELECT
GRANT USAGE ON CATALOG sales TO `analyst@company.com`;
GRANT USAGE ON SCHEMA sales.bronze TO `analyst@company.com`;
GRANT SELECT ON TABLE sales.bronze.customers TO `analyst@company.com`;
```

**Para certificação:** USAGE sozinho não dá acesso a dados — apenas permite "entrar" no container.

---

## Principals (Quem Recebe Permissões)

### Tipos de Principals

| Principal | Descrição | Sintaxe |
|-----------|-----------|---------|
| **User** | Utilizador individual | `user@company.com` |
| **Group** | Grupo de utilizadores | `data_engineers` |
| **Service Principal** | Identidade de aplicação | `app-id-12345` |

### Grupos Especiais

| Grupo | Descrição |
|-------|-----------|
| **account users** | Todos os utilizadores da conta |
| **admins** | Administradores do workspace |

**Exemplo:**

```sql
-- Dar acesso a todos os utilizadores
GRANT SELECT ON TABLE sales.gold.summary TO `account users`;
```

---

## Ownership (Propriedade)

### Regras de Ownership

| Regra | Descrição |
|-------|-----------|
| Criador é owner | Quem cria o objecto torna-se owner |
| Owner tem ALL PRIVILEGES | Owner pode fazer qualquer operação |
| Owner pode transferir | Ownership pode ser transferido |
| Apenas 1 owner | Cada objecto tem exactamente 1 owner |

### Transferir Ownership

```sql
-- Transferir ownership de tabela
ALTER TABLE sales.bronze.customers SET OWNER TO `new_owner@company.com`;

-- Transferir ownership de schema
ALTER SCHEMA sales.bronze SET OWNER TO `data_team`;
```

### Hierarquia de Quem Pode Conceder

| Role | Pode conceder em |
|------|------------------|
| **Metastore Admin** | Todos os objectos |
| **Catalog Owner** | Objectos dentro do catalog |
| **Schema Owner** | Objectos dentro do schema |
| **Table Owner** | Apenas na própria tabela |

---
### Casos de Uso Legítimos

- Ingestão de dados raw (CSV, JSON, Parquet)
- ETL que precisa aceder a ficheiros antes de criar tabelas
- Administração de storage

---
## Information Schema

### Consultar Permissões Programaticamente

```sql
-- Ver todas as tabelas com permissões
SELECT * FROM system.information_schema.table_privileges
WHERE grantee = 'user@company.com';

-- Ver grants num schema
SELECT * FROM system.information_schema.schema_privileges
WHERE schema_name = 'bronze';
```

## Boas Práticas

### Princípio do Menor Privilégio

| Fazer | Evitar |
|-------|--------|
| ✅ Conceder permissões específicas | ❌ ALL PRIVILEGES por padrão |
| ✅ Usar grupos | ❌ Permissões individuais |
| ✅ USAGE apenas onde necessário | ❌ USAGE em todo o catalog |
| ✅ Views para dados sensíveis | ❌ Acesso directo a tabelas com PII |

### Estrutura Recomendada de Grupos

```
data_readers      → SELECT em gold
data_engineers    → MODIFY em bronze/silver, SELECT em gold  
data_admins       → ALL PRIVILEGES
pii_readers       → Acesso a dados sensíveis (via views)
```

### Auditoria Regular

```sql
-- Verificar quem tem acesso a tabela sensível
SHOW GRANTS ON TABLE hr.employees;

-- Verificar permissões de um grupo
SHOW GRANTS TO `data_engineers`;
```

---

## Pontos Críticos para Certificação

### Regras Fundamentais

- ✅ **USAGE é obrigatório** para aceder a objectos em catalog/schema
- ✅ **DENY tem precedência** sobre GRANT
- ✅ **Owner tem ALL PRIVILEGES** automaticamente
- ✅ **Namespace completo**: catalog.schema.object

### Cenários de Exame

1. **Cenário:** "Utilizador tem SELECT na tabela mas não consegue aceder."
   **Resposta:** Falta USAGE no catalog ou schema

2. **Cenário:** "Quer que analistas vejam dados mas não modifiquem."
   **Resposta:** GRANT SELECT (não MODIFY)

3. **Cenário:** "Utilizador pertence a grupo com SELECT, mas não consegue aceder."
   **Resposta:** Verificar se existe DENY no utilizador ou noutro grupo

4. **Cenário:** "Quer ver todas as permissões numa tabela."
   **Resposta:** SHOW GRANTS ON TABLE catalog.schema.table

5. **Cenário:** "Novo colaborador precisa criar tabelas no schema bronze."
   **Resposta:** GRANT USAGE ON CATALOG, GRANT USAGE ON SCHEMA, GRANT CREATE ON SCHEMA

6. **Cenário:** "Transferir ownership de tabela para outro utilizador."
   **Resposta:** ALTER TABLE ... SET OWNER TO ...
---

## Comandos Resumo

```sql
-- Conceder
GRANT SELECT ON TABLE catalog.schema.table TO `user@company.com`;

-- Revogar
REVOKE SELECT ON TABLE catalog.schema.table FROM `user@company.com`;

-- Negar explicitamente
DENY MODIFY ON TABLE catalog.schema.table TO `user@company.com`;

-- Ver permissões
SHOW GRANTS ON TABLE catalog.schema.table;
SHOW GRANTS TO `user@company.com`;

-- Transferir ownership
ALTER TABLE catalog.schema.table SET OWNER TO `new_owner`;

-- Verificar utilizador actual
SELECT current_user();

-- Verificar membership
SELECT is_account_group_member('data_engineers');
```

---

## Referências Oficiais

- [Unity Catalog Privileges](https://learn.microsoft.com/pt-pt/azure/databricks/data-governance/unity-catalog/manage-privileges/)
- [GRANT Statement](https://learn.microsoft.com/pt-pt/azure/databricks/sql/language-manual/security-grant)
- [Dynamic Views](https://learn.microsoft.com/pt-pt/azure/databricks/data-governance/unity-catalog/create-views)
- [Information Schema](https://learn.microsoft.com/pt-pt/azure/databricks/sql/language-manual/information-schema)